##### Short cohort: modulated VBM

In [2]:
import os
import re
import shutil
import subprocess
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path
import ants
import nibabel as nib
import numpy as np
import pandas as pd
from tqdm import tqdm

os.environ.setdefault("FSLDIR", "/Users/william.wakefield/fsl")
_fsl_bin = f"{os.environ['FSLDIR']}/share/fsl/bin"
if _fsl_bin not in os.environ.get("PATH", "").split(os.pathsep):
    os.environ["PATH"] = _fsl_bin + os.pathsep + os.environ.get("PATH", "")
os.environ.setdefault("FSLOUTPUTTYPE", "NIFTI_GZ")

'NIFTI_GZ'

In [3]:
SHORT_T1 = Path("model_data/adni/t1_adc_short_data")
SHORT_DTI = Path("model_data/adni/dti_adc_short_data")
T1_DCM = SHORT_T1 / "t1_mri_dcm"
DTI_DCM = SHORT_DTI / "dti_dcm"
T1_RAW = SHORT_T1 / "t1_mri_raw"
DTI_RAW = SHORT_DTI / "dti_raw"
T1_VBM = SHORT_T1 / "t1_short_modulated_vbm"
DTI_VBM = SHORT_DTI / "dti_short_modulated_vbm"

T1_STRIP = T1_VBM / "t1_short_skull_strip"
T1_SEG = T1_VBM / "t1_short_seg_native"
T1_WARP = T1_VBM / "t1_short_vbm_warps"
T1_JAC = T1_VBM / "t1_short_jacobian"
T1_PVE_MNI = T1_VBM / "t1_short_pve_mni"
T1_PVE_MOD = T1_VBM / "t1_short_pve_mni_modulated"
T1_PVE_SMOOTH = T1_VBM / "t1_short_pve_mni_smoothed"

ADC_T1 = DTI_VBM / "dti_short_reg_t1_native"
ADC_MNI = DTI_VBM / "dti_short_mni"
ADC_SMOOTH = DTI_VBM / "dti_short_mni_smoothed"

MNI_T1 = Path("model_data/mni_t1.nii")

for d in (
    T1_RAW, DTI_RAW, T1_STRIP, T1_SEG, T1_WARP, T1_JAC,
    T1_PVE_MNI, T1_PVE_MOD, T1_PVE_SMOOTH,
    ADC_T1, ADC_MNI, ADC_SMOOTH,
):
    d.mkdir(parents=True, exist_ok=True)

_IMAGE_DIR = re.compile(r"^I\d+$\Z")

In [3]:
def short_image_scan_dirs(dcm_root: Path) -> list[Path]:
    """Folders named I<digits> (ADNI image UID) that contain DICOMs.
    Typical layout: .../PTID/SeriesName/DateTime/I<id>/*.dcm — depth varies, so we rglob.
    """
    if not dcm_root.is_dir():
        return []
    dirs: list[Path] = []
    for p in sorted(dcm_root.rglob("*")):
        if not p.is_dir() or not _IMAGE_DIR.fullmatch(p.name):
            continue
        if any(p.glob("*.dcm")) or any(p.glob("*.DCM")):
            dirs.append(p)
    return dirs


def subject_id_from_image_dir(image_dir: Path) -> str:
    return image_dir.parent.parent.parent.name


print("FSLOUTPUTTYPE:", os.environ.get("FSLOUTPUTTYPE"))

FSLOUTPUTTYPE: NIFTI_GZ


In [7]:
# --- DICOM → NIfTI (T1): dcm2niix ----------------------------------------------
scan_dirs = short_image_scan_dirs(T1_DCM)
print(f"Found {len(scan_dirs)} T1 image folders")

failed = []
skipped = 0

for scan_dir in tqdm(scan_dirs, desc="dcm2niix T1"):
    subj = subject_id_from_image_dir(scan_dir)
    out_name = f"{scan_dir.name}_{subj}"
    if (T1_RAW / f"{out_name}.nii").exists():
        skipped += 1
        continue
    r = subprocess.run(
        ["dcm2niix", "-z", "n", "-f", out_name, "-o", str(T1_RAW), "-b", "n", str(scan_dir)],
        capture_output=True,
        text=True,
    )
    if r.returncode != 0:
        failed.append((out_name, r.stderr.strip()))

nii = list(T1_RAW.glob("I*.nii"))
print(f"T1 NIfTI: {len(nii)} files ({skipped} skipped, {len(failed)} failed)")
if failed:
    for name, err in failed[:5]:
        print(f"  {name}: {err}")

Found 295 T1 image folders


dcm2niix T1: 100%|██████████| 295/295 [00:59<00:00,  4.95it/s]

T1 NIfTI: 295 files (0 skipped, 1 failed)
  I820315_016_S_4952: 


In [8]:
# --- DICOM → NIfTI (ADC / DTI-derived map): dcm2niix --------------------------
scan_dirs = short_image_scan_dirs(DTI_DCM)
print(f"Found {len(scan_dirs)} DTI/ADC image folders")

failed = []
skipped = 0

for scan_dir in tqdm(scan_dirs, desc="dcm2niix ADC"):
    subj = subject_id_from_image_dir(scan_dir)
    out_name = f"{scan_dir.name}_{subj}"
    if (DTI_RAW / f"{out_name}.nii").exists():
        skipped += 1
        continue
    r = subprocess.run(
        ["dcm2niix", "-z", "n", "-f", out_name, "-o", str(DTI_RAW), "-b", "n", str(scan_dir)],
        capture_output=True,
        text=True,
    )
    if r.returncode != 0:
        failed.append((out_name, r.stderr.strip()))

nii = list(DTI_RAW.glob("I*.nii"))
print(f"ADC NIfTI: {len(nii)} files ({skipped} skipped, {len(failed)} failed)")
if failed:
    for name, err in failed[:5]:
        print(f"  {name}: {err}")

Found 299 DTI/ADC image folders


dcm2niix ADC: 100%|██████████| 299/299 [00:33<00:00,  8.99it/s]

ADC NIfTI: 299 files (0 skipped, 0 failed)


In [6]:
# --- BET (native T1) ------------------------------------------------------------
# Skull remnants in FAST usually mean BET still included scalp/skull edge.
#   - `-B`: bias-field + neck cleanup (often reduces neck/skull vs `-R` alone).
#   - `-f`: FSL BET2: SMALLER `-f` = LARGER mask (more periphery).
#     If skull leaks in → raise BET_FRAC (e.g. 0.62–0.75). If cortex erodes → lower it.
# Re-run BET + FAST (and downstream) after tuning; delete old strip outputs first.
bet_frac = os.environ.get("BET_FRAC", "0.58")
raw_files = sorted(T1_RAW.glob("I*.nii"))
print(f"Skull-strip {len(raw_files)} T1 volumes (bet -R -B -f {bet_frac})")


def _run_bet(raw_path: Path, out_nii: Path):
    cmd = [
        "bet", str(raw_path), str(out_nii),
        "-R", "-B", "-f", bet_frac,
    ]
    r = subprocess.run(cmd, capture_output=True, text=True)
    return raw_path.stem, r.returncode, r.stderr.strip()


failed = []
skipped = 0
tasks = []
for raw_path in raw_files:
    out_nii = T1_STRIP / f"{raw_path.stem}.nii.gz"
    if out_nii.exists():
        skipped += 1
        continue
    tasks.append((raw_path, out_nii))

bet_workers = int(os.environ["BET_MAX_WORKERS"]) if os.environ.get("BET_MAX_WORKERS") else (os.cpu_count() or 8)
max_workers = max(1, min(len(tasks) or 1, bet_workers))
print(f"BET: {len(tasks)} jobs, {max_workers} workers ({skipped} already done)")

with ThreadPoolExecutor(max_workers=max_workers) as ex:
    futs = [ex.submit(_run_bet, a, b) for a, b in tasks]
    for fut in tqdm(as_completed(futs), total=len(futs), desc="BET"):
        stem, rc, err = fut.result()
        if rc != 0:
            failed.append((stem, err))

print(f"Stripped: {len(list(T1_STRIP.glob('*.nii.gz')))} | failed: {len(failed)}")

Skull-strip 295 T1 volumes (bet -R -B -f 0.58)
BET: 1 jobs, 1 workers (294 already done)


BET:   0%|          | 0/1 [00:00<?, ?it/s]

BET: 100%|██████████| 1/1 [00:10<00:00, 10.35s/it]

Stripped: 589 | failed: 1


In [4]:
# --- FAST (native T1) -----------------------------------------------------------
# Speed: use all CPU cores by default (was: cpu_count − 2). Cap with FAST_MAX_WORKERS.
# Extra speed (rougher PVEs): export FAST_QUICK=1  →  -I 2 -O 2 -W 8
# Or tune: FAST_I / FAST_O / FAST_W (FSL defaults: 4 / 4 / 15).
# BET also writes companion files (e.g. *_mask.nii.gz). Only run FAST on the
# brain-extracted volume that matches each raw stem: {stem}.nii.gz — not masks.
_brain_paths = [T1_STRIP / f"{raw_path.stem}.nii.gz" for raw_path in sorted(T1_RAW.glob("I*.nii"))]
stripped = [p for p in _brain_paths if p.exists()]
_n_missing = len(_brain_paths) - len(stripped)
print(
    f"FAST on {len(stripped)} brain-extracted T1s "
    f"({_n_missing} without strip yet; skipped *_mask.nii.gz and other BET sidecars)"
)

if os.environ.get("FAST_QUICK", "").lower() in ("1", "true", "yes"):
    _I, _O, _W = "2", "2", "8"
else:
    _I = os.environ.get("FAST_I", "4")
    _O = os.environ.get("FAST_O", "4")
    _W = os.environ.get("FAST_W", "15")


def _run_fast(strip_path: Path, out_base: Path):
    r = subprocess.run(
        [
            "fast", "-t", "1", "-n", "3", "-B", "-b",
            "-I", _I, "-O", _O, "-W", _W,
            "-o", str(out_base), str(strip_path),
        ],
        capture_output=True,
        text=True,
    )
    stem = strip_path.name.replace(".nii.gz", "")
    return stem, r.returncode, r.stderr.strip()


failed = []
skipped = 0
tasks = []
for strip_path in stripped:
    stem = strip_path.name.replace(".nii.gz", "")
    subj_dir = T1_SEG / stem
    subj_dir.mkdir(parents=True, exist_ok=True)
    out_base = subj_dir / stem
    pve = [subj_dir / f"{stem}_pve_{i}.nii.gz" for i in range(3)]
    if all(p.exists() for p in pve):
        skipped += 1
        continue
    tasks.append((strip_path, out_base))

_n = int(os.environ["FAST_MAX_WORKERS"]) if os.environ.get("FAST_MAX_WORKERS") else (os.cpu_count() or 8)
max_workers = max(1, min(len(tasks) or 1, _n))
print(
    f"FAST: {len(tasks)} jobs ({skipped} skipped) | workers={max_workers} | "
    f"-I {_I} -O {_O} -W {_W}"
)

with ThreadPoolExecutor(max_workers=max_workers) as ex:
    futs = [ex.submit(_run_fast, a, b) for a, b in tasks]
    for fut in tqdm(as_completed(futs), total=len(futs), desc="FAST"):
        stem, rc, err = fut.result()
        if rc != 0:
            failed.append((stem, err))

csf = list(T1_SEG.glob("*/*_pve_0.nii.gz"))
print(f"PVE maps: {len(csf)} subjects | FAST failed: {len(failed)}")

FAST on 294 brain-extracted T1s (1 without strip yet; skipped *_mask.nii.gz and other BET sidecars)
FAST: 294 jobs (0 skipped) | workers=16 | -I 4 -O 4 -W 15


FAST: 100%|██████████| 294/294 [3:12:22<00:00, 39.26s/it]     

PVE maps: 294 subjects | FAST failed: 0


In [5]:
# --- SyN: raw T1 → MNI (forward warps for VBM) --------------------------------
mni_t1 = ants.image_read(str(MNI_T1))
raw_files = sorted(T1_RAW.glob("I*.nii"))
print(f"Register {len(raw_files)} native T1 → MNI")

failed = []
skipped = 0
for raw_path in tqdm(raw_files, desc="SyN T1→MNI"):
    stem = raw_path.stem
    out_warped = T1_WARP / f"{stem}_warpedT1.nii.gz"
    out_affine = T1_WARP / f"{stem}_0GenericAffine.mat"
    out_warp = T1_WARP / f"{stem}_1Warp.nii.gz"
    out_inv = T1_WARP / f"{stem}_1InverseWarp.nii.gz"
    if out_warped.exists() and out_affine.exists() and out_warp.exists() and out_inv.exists():
        skipped += 1
        continue
    try:
        moving = ants.image_read(str(raw_path))
        reg = ants.registration(fixed=mni_t1, moving=moving, type_of_transform="SyN")
        ants.image_write(reg["warpedmovout"], str(out_warped))
        for tx_path in reg["fwdtransforms"]:
            tx = Path(tx_path)
            if "GenericAffine" in tx.name:
                shutil.copy(tx, out_affine)
            elif "Warp" in tx.name and "Inverse" not in tx.name:
                shutil.copy(tx, out_warp)
        for tx_path in reg["invtransforms"]:
            tx = Path(tx_path)
            if "InverseWarp" in tx.name:
                shutil.copy(tx, out_inv)
    except Exception as e:
        failed.append((stem, str(e)))

print(f"Warps: {len(list(T1_WARP.glob('*_1Warp.nii.gz')))} | skipped {skipped} | failed {len(failed)}")

Register 295 native T1 → MNI


SyN T1→MNI: 100%|██████████| 295/295 [18:02<00:00,  3.67s/it]

Warps: 295 | skipped 0 | failed 0


In [6]:
# --- Jacobian det (from SyN warp + affine) ------------------------------------
warp_files = sorted(T1_WARP.glob("*_1Warp.nii.gz"))
print(f"Jacobians for {len(warp_files)} warps")

failed = []
skipped = 0

for warp_path in tqdm(warp_files, desc="Jacobian"):
    stem = warp_path.name.replace("_1Warp.nii.gz", "")
    affine_path = T1_WARP / f"{stem}_0GenericAffine.mat"
    out_jac = T1_JAC / f"{stem}_jacobian.nii.gz"
    if out_jac.exists():
        skipped += 1
        continue
    if not affine_path.exists():
        failed.append((stem, "missing affine"))
        continue
    try:
        jac_warp = ants.create_jacobian_determinant_image(
            domain_image=mni_t1, tx=str(warp_path), do_log=False, geom=False,
        )
        affine_tx = ants.read_transform(str(affine_path))
        params = np.asarray(affine_tx.parameters)
        det_affine = abs(float(np.linalg.det(params[:9].reshape(3, 3))))
        jac_total = jac_warp * det_affine
        ants.image_write(jac_total, str(out_jac))
    except Exception as e:
        failed.append((stem, str(e)))

print(f"Jacobian maps: {len(list(T1_JAC.glob('*_jacobian.nii.gz')))} | failed {len(failed)}")

Jacobians for 295 warps


Jacobian: 100%|██████████| 295/295 [01:11<00:00,  4.13it/s]

Jacobian maps: 295 | failed 0


In [7]:
# --- Warp native PVEs → MNI ----------------------------------------------------
subject_dirs = sorted(p for p in T1_SEG.iterdir() if p.is_dir())
print(f"Warp PVEs for {len(subject_dirs)} subjects")

failed = []
skipped = 0

for subj_dir in tqdm(subject_dirs, desc="PVE→MNI"):
    stem = subj_dir.name
    out_sub = T1_PVE_MNI / stem
    out_sub.mkdir(parents=True, exist_ok=True)
    affine = T1_WARP / f"{stem}_0GenericAffine.mat"
    warp = T1_WARP / f"{stem}_1Warp.nii.gz"
    if not affine.exists() or not warp.exists():
        failed.append((stem, "missing transforms"))
        continue
    tx = [str(warp), str(affine)]
    if all((out_sub / f"{stem}_pve_{i}_mni.nii.gz").exists() for i in range(3)):
        skipped += 1
        continue
    try:
        for i in range(3):
            in_pve = subj_dir / f"{stem}_pve_{i}.nii.gz"
            out_pve = out_sub / f"{stem}_pve_{i}_mni.nii.gz"
            if out_pve.exists():
                continue
            if not in_pve.exists():
                failed.append((stem, f"missing pve_{i}"))
                break
            moving = ants.image_read(str(in_pve))
            warped = ants.apply_transforms(
                fixed=mni_t1, moving=moving, transformlist=tx, interpolator="linear",
            )
            ants.image_write(warped, str(out_pve))
    except Exception as e:
        failed.append((stem, str(e)))

print(f"MNI PVE dirs: {len(list(T1_PVE_MNI.iterdir()))} | failed {len(failed)}")

Warp PVEs for 294 subjects


PVE→MNI: 100%|██████████| 294/294 [03:00<00:00,  1.63it/s]

MNI PVE dirs: 295 | failed 0


In [8]:
# --- Modulate MNI PVE × Jacobian ------------------------------------------------
subject_dirs = sorted(p for p in T1_PVE_MNI.iterdir() if p.is_dir())
print(f"Modulate {len(subject_dirs)} subjects")

failed = []
skipped = 0

for subj_dir in tqdm(subject_dirs, desc="Modulate"):
    stem = subj_dir.name
    out_sub = T1_PVE_MOD / stem
    out_sub.mkdir(parents=True, exist_ok=True)
    jac_path = T1_JAC / f"{stem}_jacobian.nii.gz"
    if not jac_path.exists():
        failed.append((stem, "missing jacobian"))
        continue
    if all((out_sub / f"{stem}_pve_{i}_mod.nii.gz").exists() for i in range(3)):
        skipped += 1
        continue
    try:
        jac_img = ants.image_read(str(jac_path))
        jac_arr = jac_img.numpy()
        for i in range(3):
            in_pve = subj_dir / f"{stem}_pve_{i}_mni.nii.gz"
            out_pve = out_sub / f"{stem}_pve_{i}_mod.nii.gz"
            if out_pve.exists():
                continue
            if not in_pve.exists():
                failed.append((stem, f"missing pve_{i}_mni"))
                continue
            pve_img = ants.image_read(str(in_pve))
            mod_arr = pve_img.numpy() * jac_arr
            mod_img = ants.from_numpy(
                mod_arr, origin=pve_img.origin,
                spacing=pve_img.spacing, direction=pve_img.direction,
            )
            ants.image_write(mod_img, str(out_pve))
    except Exception as e:
        failed.append((stem, str(e)))

print(f"Modulated: {len(list(T1_PVE_MOD.glob('*/*_pve_0_mod.nii.gz')))} CSF-like | skipped {skipped} | failures {len(failed)}")

Modulate 294 subjects


Modulate: 100%|██████████| 294/294 [00:40<00:00,  7.23it/s]

Modulated: 294 CSF-like | skipped 0 | failures 0


In [9]:
# --- Smooth modulated PVE (2.5 mm FWHM) -----------------------------------------
fwhm_mm = 2.5
sigma_mm = fwhm_mm / (2.0 * np.sqrt(2.0 * np.log(2.0)))
subject_dirs = sorted(p for p in T1_PVE_MOD.iterdir() if p.is_dir())
print(f"Smooth {len(subject_dirs)} subjects @ {fwhm_mm} mm FWHM")

failed = []
skipped = 0

for subj_dir in tqdm(subject_dirs, desc="Smooth"):
    stem = subj_dir.name
    out_sub = T1_PVE_SMOOTH / stem
    out_sub.mkdir(parents=True, exist_ok=True)
    if all((out_sub / f"{stem}_pve_{i}_mod_s2p5.nii.gz").exists() for i in range(3)):
        skipped += 1
        continue
    try:
        for i in range(3):
            in_pve = subj_dir / f"{stem}_pve_{i}_mod.nii.gz"
            out_pve = out_sub / f"{stem}_pve_{i}_mod_s2p5.nii.gz"
            if out_pve.exists():
                continue
            if not in_pve.exists():
                failed.append((stem, f"missing pve_{i}_mod"))
                continue
            img = ants.image_read(str(in_pve))
            sm = ants.smooth_image(
                img, sigma=sigma_mm, sigma_in_physical_coordinates=True,
            )
            ants.image_write(sm, str(out_pve))
    except Exception as e:
        failed.append((stem, str(e)))

print(
    "Smoothed:", len(list(T1_PVE_SMOOTH.glob("*/*_pve_1_mod_s2p5.nii.gz"))),
    "GM | failures:", len(failed),
)

Smooth 294 subjects @ 2.5 mm FWHM


Smooth: 100%|██████████| 294/294 [00:46<00:00,  6.30it/s]

Smoothed: 294 GM | failures: 0


In [10]:
# --- Map each ADC volume → matching skull-stripped T1 (by subject, scan order) -
subj_pat = re.compile(r"^I\d+_(\d{3}_S_\d+)$")

t1_by_subj: dict[str, list[str]] = {}
for f in sorted(T1_STRIP.glob("*.nii.gz")):
    stem = f.name.replace(".nii.gz", "")
    m = subj_pat.match(stem)
    if m:
        t1_by_subj.setdefault(m.group(1), []).append(stem)

adc_by_subj: dict[str, list[tuple[str, Path]]] = {}
adc_paths = sorted(DTI_RAW.glob("I*.nii"))
for p in adc_paths:
    stem = p.stem
    m = subj_pat.match(stem)
    if m:
        adc_by_subj.setdefault(m.group(1), []).append((stem, p))

dti_to_t1: dict[str, str] = {}
for subj, adc_list in adc_by_subj.items():
    t1_list = sorted(t1_by_subj.get(subj, []))
    adc_sorted = sorted(adc_list, key=lambda x: x[0])
    for i, (adc_stem, _) in enumerate(adc_sorted):
        if not t1_list:
            continue
        t1_stem = t1_list[i] if i < len(t1_list) else t1_list[-1]
        dti_to_t1[adc_stem] = t1_stem

print(f"ADC files: {len(adc_paths)} | matched to T1: {len(dti_to_t1)}")

ADC files: 299 | matched to T1: 299


In [11]:
# --- Rigid: ADC → native T1 space (no FA; register ADC to T1 directly) --------
def _rigid_adc_to_t1(adc_path: Path):
    adc_stem = adc_path.stem
    t1_stem = dti_to_t1.get(adc_stem)
    out_adc = ADC_T1 / f"{adc_stem}_ADC.nii.gz"
    out_tx = ADC_T1 / f"{adc_stem}_rigid.mat"
    if out_adc.exists() and out_tx.exists():
        return adc_stem, "skip", ""
    if t1_stem is None:
        return adc_stem, "fail", "no T1 match"
    t1_path = T1_STRIP / f"{t1_stem}.nii.gz"
    if not t1_path.exists():
        return adc_stem, "fail", f"missing {t1_path.name}"
    try:
        fixed = ants.image_read(str(t1_path))
        moving = ants.image_read(str(adc_path))
        reg = ants.registration(fixed=fixed, moving=moving, type_of_transform="Rigid")
        adc_in_t1 = ants.apply_transforms(
            fixed=fixed,
            moving=moving,
            transformlist=reg["fwdtransforms"],
            interpolator="linear",
        )
        ants.image_write(adc_in_t1, str(out_adc))
        for tx_path in reg["fwdtransforms"]:
            if Path(tx_path).suffix == ".mat" or "GenericAffine" in Path(tx_path).name:
                shutil.copy(tx_path, out_tx)
                break
        return adc_stem, "ok", ""
    except Exception as e:
        return adc_stem, "fail", str(e)


failed = []
skipped = ok = 0
max_workers = max(1, (os.cpu_count() or 4) - 2)
adc_to_reg = [p for p in adc_paths if dti_to_t1.get(p.stem)]
print(f"Rigid ADC→T1: {len(adc_to_reg)} maps, {max_workers} workers")

with ThreadPoolExecutor(max_workers=max_workers) as ex:
    futs = [ex.submit(_rigid_adc_to_t1, p) for p in adc_to_reg]
    for fut in tqdm(as_completed(futs), total=len(futs), desc="Rigid ADC"):
        stem, status, err = fut.result()
        if status == "ok":
            ok += 1
        elif status == "skip":
            skipped += 1
        else:
            failed.append((stem, err))

print(f"ADC in T1 space: {len(list(ADC_T1.glob('*_ADC.nii.gz')))} | ok {ok} skip {skipped} fail {len(failed)}")

Rigid ADC→T1: 299 maps, 14 workers


Rigid ADC: 100%|██████████| 299/299 [14:52<00:00,  2.99s/it]

ADC in T1 space: 297 | ok 297 skip 0 fail 2


In [12]:
# --- Warp ADC (T1 space) → MNI using same SyN warps as T1 ---------------------
md_t1_files = sorted(ADC_T1.glob("*_ADC.nii.gz"))
print(f"Apply T1 warps to {len(md_t1_files)} ADC maps")


def _warp_adc_to_mni(md_t1_path: Path):
    adc_stem = md_t1_path.name.replace("_ADC.nii.gz", "")
    t1_stem = dti_to_t1.get(adc_stem)
    out_adc = ADC_MNI / f"{adc_stem}_ADC.nii.gz"
    if out_adc.exists():
        return adc_stem, "skip", ""
    if t1_stem is None:
        return adc_stem, "fail", "no T1 map"
    affine = T1_WARP / f"{t1_stem}_0GenericAffine.mat"
    warp = T1_WARP / f"{t1_stem}_1Warp.nii.gz"
    if not affine.exists() or not warp.exists():
        return adc_stem, "fail", "missing warps"
    try:
        moving = ants.image_read(str(md_t1_path))
        warped = ants.apply_transforms(
            fixed=mni_t1,
            moving=moving,
            transformlist=[str(warp), str(affine)],
            interpolator="linear",
        )
        ants.image_write(warped, str(out_adc))
        return adc_stem, "ok", ""
    except Exception as e:
        return adc_stem, "fail", str(e)


failed = []
skipped = ok = 0
max_workers = max(1, (os.cpu_count() or 4) - 2)
with ThreadPoolExecutor(max_workers=max_workers) as ex:
    futs = [ex.submit(_warp_adc_to_mni, p) for p in md_t1_files]
    for fut in tqdm(as_completed(futs), total=len(futs), desc="ADC→MNI"):
        stem, status, err = fut.result()
        if status == "ok":
            ok += 1
        elif status == "skip":
            skipped += 1
        else:
            failed.append((stem, err))

print(f"MNI ADC: {len(list(ADC_MNI.glob('*_ADC.nii.gz')))} | failed {len(failed)}")

Apply T1 warps to 297 ADC maps


ADC→MNI: 100%|██████████| 297/297 [02:37<00:00,  1.89it/s]

MNI ADC: 297 | failed 0


In [13]:
# --- Smooth MNI ADC (5 mm FWHM), match long-MD preprocessing -----------------
fwhm_mm = 5.0
sigma_mm = fwhm_mm / (2.0 * np.sqrt(2.0 * np.log(2.0)))
mni_adc = sorted(ADC_MNI.glob("*_ADC.nii.gz"))
print(f"Smooth {len(mni_adc)} ADC volumes @ {fwhm_mm} mm FWHM")


def _smooth_adc(pth: Path):
    adc_stem = pth.name.replace("_ADC.nii.gz", "")
    out_path = ADC_SMOOTH / f"{adc_stem}_ADC_s5.nii.gz"
    if out_path.exists():
        return adc_stem, "skip", ""
    try:
        img = ants.image_read(str(pth))
        sm = ants.smooth_image(
            img, sigma=sigma_mm, sigma_in_physical_coordinates=True,
        )
        ants.image_write(sm, str(out_path))
        return adc_stem, "ok", ""
    except Exception as e:
        return adc_stem, "fail", str(e)


failed = []
skipped = ok = 0
max_workers = max(1, (os.cpu_count() or 4) - 2)
with ThreadPoolExecutor(max_workers=max_workers) as ex:
    futs = [ex.submit(_smooth_adc, p) for p in mni_adc]
    for fut in tqdm(as_completed(futs), total=len(futs), desc="Smooth ADC"):
        stem, status, err = fut.result()
        if status == "ok":
            ok += 1
        elif status == "skip":
            skipped += 1
        else:
            failed.append((stem, err))

print(f"Smoothed ADC: {len(list(ADC_SMOOTH.glob('*_ADC_s5.nii.gz')))} | failed {len(failed)}")

Smooth 297 ADC volumes @ 5.0 mm FWHM


Smooth ADC: 100%|██████████| 297/297 [00:48<00:00,  6.12it/s]

Smoothed ADC: 297 | failed 0


#### ML dataset preprocessing (tissue masks → parquets + `paired_df_short.csv`)

Dx from **DXSUM** (`DIAGNOSIS` 1/2/3). Amyloid from UC Berkeley spreadsheet.


In [14]:
# --- Resample MNI tissue masks to smoothed ADC grid ----------------------------
import nilearn.image as nli

adc_smooth_files = sorted(ADC_SMOOTH.glob("*_ADC_s5.nii.gz"))
if not adc_smooth_files:
    raise FileNotFoundError("No smoothed ADC maps; run prior cells.")

ref_img = nib.load(adc_smooth_files[0])


def _load_mask(path: str):
    m = nli.resample_to_img(
        path, ref_img, interpolation="nearest", force_resample=True, copy_header=True,
    )
    return np.asarray(m.dataobj) > 0.5


gm_mask = _load_mask("model_data/mni_gm_mask.nii")
wm_mask = _load_mask("model_data/mni_wm_mask.nii")
csf_mask = _load_mask("model_data/mni_csf_mask.nii")
print(
    f"Ref shape {ref_img.shape} | GM {gm_mask.sum():,} | WM {wm_mask.sum():,} | CSF {csf_mask.sum():,}"
)

Ref shape (121, 145, 121) | GM 451,681 | WM 283,320 | CSF 173,822


In [15]:
# --- Pair T1 ↔ ADC by subject + scan order; write T1 parquets ----------------


def _parse(stem: str):
    img, _, subj = stem.partition("_")
    return img.lstrip("I"), subj


t1_subjects = sorted(p for p in T1_PVE_SMOOTH.iterdir() if p.is_dir())
dti_files = sorted(ADC_SMOOTH.glob("*_ADC_s5.nii.gz"))
t1_all = pd.DataFrame(
    [(p.name, *_parse(p.name)) for p in t1_subjects],
    columns=["t1_image_subject_id", "t1_image_id", "subject_id"],
)
dti_all = pd.DataFrame(
    [(s := f.name.replace("_ADC_s5.nii.gz", ""), *_parse(s)) for f in dti_files],
    columns=["dti_image_subject_id", "dti_image_id", "subject_id"],
)
t1_all = t1_all.sort_values(["subject_id", "t1_image_id"]).reset_index(drop=True)
dti_all = dti_all.sort_values(["subject_id", "dti_image_id"]).reset_index(drop=True)
t1_all["scan_n"] = t1_all.groupby("subject_id").cumcount()
dti_all["scan_n"] = dti_all.groupby("subject_id").cumcount()
paired_stems = t1_all.merge(dti_all, on=["subject_id", "scan_n"], how="inner").drop(
    columns="scan_n"
)
print(f"T1 {len(t1_all)} | ADC {len(dti_all)} | paired {len(paired_stems)}")

paired_t1_dirs = [T1_PVE_SMOOTH / s for s in paired_stems["t1_image_subject_id"]]


def _extract_t1(subj_dir: Path):
    stem = subj_dir.name
    csf = nib.load(subj_dir / f"{stem}_pve_0_mod_s2p5.nii.gz").get_fdata(dtype=np.float32)
    gm = nib.load(subj_dir / f"{stem}_pve_1_mod_s2p5.nii.gz").get_fdata(dtype=np.float32)
    wm = nib.load(subj_dir / f"{stem}_pve_2_mod_s2p5.nii.gz").get_fdata(dtype=np.float32)
    return stem, gm[gm_mask], wm[wm_mask], csf[csf_mask]


stems, gm_rows, wm_rows, csf_rows = [], [], [], []
with ThreadPoolExecutor(max_workers=max(1, (os.cpu_count() or 4) - 2)) as ex:
    for stem, g, w, c in tqdm(
        ex.map(_extract_t1, paired_t1_dirs),
        total=len(paired_t1_dirs),
        desc="T1 parquets",
    ):
        stems.append(stem)
        gm_rows.append(g)
        wm_rows.append(w)
        csf_rows.append(c)

ix = pd.Index(stems, name="image_subject_id")
gm_df = pd.DataFrame(np.vstack(gm_rows), index=ix)
wm_df = pd.DataFrame(np.vstack(wm_rows), index=ix)
csf_df = pd.DataFrame(np.vstack(csf_rows), index=ix)
for df in (gm_df, wm_df, csf_df):
    df.columns = df.columns.astype(str)

gm_df.to_parquet(SHORT_T1 / "t1_short_masked_gm.parquet", compression="zstd")
wm_df.to_parquet(SHORT_T1 / "t1_short_masked_wm.parquet", compression="zstd")
csf_df.to_parquet(SHORT_T1 / "t1_short_masked_csf.parquet", compression="zstd")
print(gm_df.shape, wm_df.shape, csf_df.shape)

T1 294 | ADC 297 | paired 292


T1 parquets: 100%|██████████| 292/292 [00:36<00:00,  8.07it/s]


(292, 451681) (292, 283320) (292, 173822)


In [16]:
# --- ADC parquets (paired rows only) -------------------------------------------


def _extract_adc(pth: Path):
    stem = pth.name.replace("_ADC_s5.nii.gz", "")
    vol = nib.load(pth).get_fdata(dtype=np.float32)
    return stem, vol[gm_mask], vol[wm_mask], vol[csf_mask]


paired_adc = [
    ADC_SMOOTH / f"{s}_ADC_s5.nii.gz" for s in paired_stems["dti_image_subject_id"]
]
paired_existing = [p for p in paired_adc if p.exists()]
missing = len(paired_adc) - len(paired_existing)
if missing:
    print("Warning: missing smoothed files:", missing)

stems, gm_rows, wm_rows, csf_rows = [], [], [], []
with ThreadPoolExecutor(max_workers=max(1, (os.cpu_count() or 4) - 2)) as ex:
    for stem, g, w, c in tqdm(
        ex.map(_extract_adc, paired_existing),
        total=len(paired_existing),
        desc="ADC parquets",
    ):
        stems.append(stem)
        gm_rows.append(g)
        wm_rows.append(w)
        csf_rows.append(c)

ix = pd.Index(stems, name="image_subject_id")
adc_gm = pd.DataFrame(np.vstack(gm_rows), index=ix)
adc_wm = pd.DataFrame(np.vstack(wm_rows), index=ix)
adc_csf = pd.DataFrame(np.vstack(csf_rows), index=ix)
for df in (adc_gm, adc_wm, adc_csf):
    df.columns = df.columns.astype(str)

adc_gm.to_parquet(SHORT_DTI / "dti_short_masked_gm_adc.parquet", compression="zstd")
adc_wm.to_parquet(SHORT_DTI / "dti_short_masked_wm_adc.parquet", compression="zstd")
adc_csf.to_parquet(SHORT_DTI / "dti_short_masked_csf_adc.parquet", compression="zstd")
print(adc_gm.shape, adc_wm.shape, adc_csf.shape)

ADC parquets: 100%|██████████| 292/292 [00:04<00:00, 66.45it/s]


(292, 451681) (292, 283320) (292, 173822)


In [17]:
# --- paired_df_short.csv (Dx from DXSUM, not ADNIMERGE) ------------------------
meta_csv = Path("model_data/adni/paired_df_short.csv")
paired = paired_stems.copy()

# DXSUM: latest DIAGNOSIS per PTID (1 = CN / normal, 2 = MCI, 3 = AD / dementia-like)
_dxsum_csv = Path("model_data/adni/DXSUM_22Sep2025.csv")
dxsum = pd.read_csv(
    _dxsum_csv,
    usecols=["PTID", "EXAMDATE", "DIAGNOSIS"],
    low_memory=False,
)
dxsum["EXAMDATE"] = pd.to_datetime(dxsum["EXAMDATE"], errors="coerce")
dxsum["DIAGNOSIS"] = pd.to_numeric(dxsum["DIAGNOSIS"], errors="coerce")
dxsum = dxsum.dropna(subset=["DIAGNOSIS"]).loc[lambda z: z["DIAGNOSIS"].isin([1, 2, 3])]
dxsum = dxsum.sort_values("EXAMDATE").drop_duplicates("PTID", keep="last")
_dx_last = dxsum.set_index("PTID")["DIAGNOSIS"].astype(int)
paired["group"] = paired["subject_id"].map(_dx_last.map({1: "CN", 2: "MCI", 3: "Dementia"}))
paired["diag_label"] = paired["subject_id"].map(_dx_last).map({1: 0, 2: 1, 3: 1})

amy = pd.read_csv(
    "model_data/adni/All_Subjects_UCBERKELEY_AMY_6MM_08Feb2026.csv",
    usecols=["PTID", "SCANDATE", "AMYLOID_STATUS"],
    low_memory=False,
)
amy = (
    amy.dropna(subset=["AMYLOID_STATUS"])
    .sort_values("SCANDATE")
    .drop_duplicates("PTID", keep="last")
)
paired["amyloid_label"] = paired["subject_id"].map(
    dict(zip(amy["PTID"], amy["AMYLOID_STATUS"]))
).astype("Int64")

paired = paired[
    [
        "subject_id",
        "t1_image_id",
        "dti_image_id",
        "t1_image_subject_id",
        "dti_image_subject_id",
        "group",
        "diag_label",
        "amyloid_label",
    ]
]
paired.to_csv(meta_csv, index=False)
print(f"Wrote {meta_csv} ({len(paired)} rows)")
print(paired["group"].value_counts(dropna=False))


Wrote model_data/adni/paired_df_short.csv (292 rows)
group
CN          149
MCI          89
Dementia     54
Name: count, dtype: int64
